# # 04 - Model 3: Denial Reason (multiclass)
#
# **Goal:** for a claim that is likely to be denied, predict the **most likely denial
# reason**, so the billing team knows what to fix before sending the claim.
#
# * Target: `denial_category` (7 classes, denied claims only).
# * Model: **XGBoost multiclass** inside a scikit-learn pipeline.
# * We never use `denial_reason_code` as an input, because that would give away the answer.
# * We use the same **5 features** as the other two models.
# * We try a few settings with RandomizedSearchCV and keep the best one.
#
# ### Note about the accuracy (about 35%)
# This score is low, and that is expected - it is not a bug:
#
# * The real denial reason is decided by the payer through the X12 `denial_reason_code`.
#   That code is **removed from the inputs on purpose** (otherwise the model would cheat).
# * The 5 features we are allowed to use (documentation completeness, CPT code, ICD-10
#   diagnosis, billed amount, auth status) do not really say **why** a claim was denied.
#   For example, the billed amount has almost the same reason mix in every price range.
# * Just guessing the biggest class gives about **28% accuracy**, so 35% is only a small
#   improvement over a blind guess.
# * We also tested giving the rare classes more weight. The macro-F1 went up only a
#   little (0.206 -> 0.214) and the accuracy went down (0.35 -> 0.26), so we keep the
#   normal model.


# ## 1. Import the libraries and set the folders


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from xgboost import XGBClassifier

RANDOM_STATE = 42

# find the project folder (the one that contains data/processed)
BASE = Path.cwd()
while not (BASE / "data" / "processed").exists() and BASE != BASE.parent:
    BASE = BASE.parent

PROCESSED = BASE / "data" / "processed"
MODELS = BASE / "models"
REPORTS = BASE / "reports"
MODELS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)


# ## 2. Load the data and keep only the denied claims
#
# Only denied claims have a denial reason, so we drop the others.


In [2]:
df = pd.read_csv(PROCESSED / "claims_model_ready.csv")

with open(PROCESSED / "feature_columns.json", encoding="utf-8") as fh:
    cols = json.load(fh)

# use only the 5 features that were chosen in notebook 01
CATEGORICAL = cols["selected_categorical"]
NUMERIC = cols["selected_numeric"]
FEATURES = cols["selected_features"]

print("Selected features:", FEATURES)

# keep only the denied claims
data = df[df["denial_category"].notna()].copy()
print("Denied claims:", data.shape)
print(data["denial_category"].value_counts())


Selected features: ['documentation_completeness', 'cpt_code', 'primary_icd10_dx', 'claim_amount_usd', 'auth_status']
Denied claims: (33664, 40)
denial_category
medical_necessity    9444
coding_error         7344
auth_missing         6140
eligibility          4084
duplicate            2645
timely_filing        2315
bundling             1692
Name: count, dtype: int64


# ## 3. Turn the text labels into numbers and split the data


In [3]:
# the 7 reason names become the numbers 0 to 6
le = LabelEncoder()
data["y"] = le.fit_transform(data["denial_category"])
classes = list(le.classes_)
print("Classes:", classes)

X = data[FEATURES]
y = data["y"]

# use the same train / val / test split that notebook 01 created
train_rows = data["split"] == "train"
val_rows = data["split"] == "val"
test_rows = data["split"] == "test"

X_train = X[train_rows]
y_train = y[train_rows]
X_val = X[val_rows]
y_val = y[val_rows]
X_test = X[test_rows]
y_test = y[test_rows]

print("train/val/test:", X_train.shape, X_val.shape, X_test.shape)


Classes: ['auth_missing', 'bundling', 'coding_error', 'duplicate', 'eligibility', 'medical_necessity', 'timely_filing']
train/val/test: (23597, 5) (4965, 5) (5102, 5)


# ## 4. Build the preprocessing pipeline
#
# Numbers are filled with the median and scaled. Text columns are filled with the most
# common value and then one-hot encoded.


In [4]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL),
])


# ## 5. Try a few settings and keep the best model


In [5]:
pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", XGBClassifier(objective="multi:softprob",
                          num_class=len(classes),
                          tree_method="hist",
                          eval_metric="mlogloss",
                          random_state=RANDOM_STATE,
                          n_jobs=1)),
])

# the settings we want to test
param_dist = {
    "clf__n_estimators": [200, 300, 400, 600],
    "clf__max_depth": [3, 4, 5, 6, 8],
    "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "clf__min_child_weight": [1, 3, 5],
    "clf__gamma": [0, 0.1, 0.3],
    "clf__reg_alpha": [0, 0.1, 1],
    "clf__reg_lambda": [1, 1.5, 2],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, scoring="f1_macro",
                            cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True)
search.fit(X_train, y_train)

print("Best CV macro-F1:", round(search.best_score_, 4))
print("Best params:", search.best_params_)


Best CV macro-F1: 0.209
Best params: {'clf__subsample': 1.0, 'clf__reg_lambda': 2, 'clf__reg_alpha': 1, 'clf__n_estimators': 300, 'clf__min_child_weight': 3, 'clf__max_depth': 8, 'clf__learning_rate': 0.2, 'clf__gamma': 0, 'clf__colsample_bytree': 0.8}


# ## 6. Check the model on the test data


In [6]:
best = search.best_estimator_
pred = best.predict(X_test)

accuracy = accuracy_score(y_test, pred)
precision_macro = precision_score(y_test, pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, pred, average="weighted", zero_division=0)

metrics = {
    "accuracy": round(accuracy, 4),
    "precision_macro": round(precision_macro, 4),
    "recall_macro": round(recall_macro, 4),
    "macro_f1": round(macro_f1, 4),
    "weighted_f1": round(weighted_f1, 4),
    "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    "classes": classes,
    "test_rows": int(len(y_test)),
}

print(json.dumps(metrics, indent=2))
print(classification_report(y_test, pred, target_names=classes, zero_division=0))


{
  "accuracy": 0.3503,
  "precision_macro": 0.2215,
  "recall_macro": 0.2266,
  "macro_f1": 0.2057,
  "weighted_f1": 0.3041,
  "confusion_matrix": [
    [
      507,
      3,
      204,
      6,
      20,
      203,
      2
    ],
    [
      34,
      0,
      60,
      4,
      11,
      140,
      2
    ],
    [
      217,
      4,
      412,
      18,
      46,
      429,
      6
    ],
    [
      53,
      2,
      74,
      13,
      28,
      224,
      9
    ],
    [
      73,
      8,
      122,
      15,
      35,
      331,
      11
    ],
    [
      185,
      11,
      286,
      25,
      73,
      814,
      14
    ],
    [
      47,
      2,
      72,
      8,
      21,
      212,
      6
    ]
  ],
  "classes": [
    "auth_missing",
    "bundling",
    "coding_error",
    "duplicate",
    "eligibility",
    "medical_necessity",
    "timely_filing"
  ],
  "test_rows": 5102
}
                   precision    recall  f1-score   support

     auth_missing       0.45     

# ## 7. Draw the confusion matrix


In [7]:
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Denial reason confusion matrix")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORTS / "04_confusion_matrix.png", dpi=110)
plt.show()


C:\Windows\Temp\ipykernel_22408\3505484816.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 8. Which features matter most


In [8]:
feat_names = best.named_steps["prep"].get_feature_names_out()
importances = best.named_steps["clf"].feature_importances_

imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False)
imp_df = imp_df.reset_index(drop=True)
print(imp_df.head(15))

plt.figure(figsize=(8, 6))
sns.barplot(data=imp_df.head(20), y="feature", x="importance")
plt.title("Denial reason model - top 20 features")
plt.tight_layout()
plt.savefig(REPORTS / "04_feature_importance.png", dpi=110)
plt.show()


                            feature  importance
0     cat__auth_status_not_obtained    0.105595
1         cat__auth_status_obtained    0.048194
2     cat__auth_status_not_required    0.032269
3   num__documentation_completeness    0.024434
4     cat__primary_icd10_dx_S72.001    0.013085
5       cat__primary_icd10_dx_A41.9    0.012750
6         cat__primary_icd10_dx_I10    0.012194
7       cat__primary_icd10_dx_M54.5    0.011933
8      cat__primary_icd10_dx_Z34.00    0.011895
9               cat__cpt_code_95886    0.011853
10      cat__primary_icd10_dx_J18.9    0.011686
11     cat__primary_icd10_dx_C34.10    0.011528
12      cat__primary_icd10_dx_I50.9    0.011478
13     cat__primary_icd10_dx_I25.10    0.011271
14      cat__primary_icd10_dx_J44.1    0.011204


C:\Windows\Temp\ipykernel_22408\3241254438.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# ## 9. Save the model and its scores
#
# We save the pipeline together with the class names, so the API can turn a predicted
# number back into a reason name.


In [9]:
joblib.dump({"pipeline": best, "classes": classes}, MODELS / "denial_reason_model.pkl")

meta_path = MODELS / "model_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
else:
    meta = {}

# json cannot store numpy numbers, so turn them into normal python numbers
best_params = {}
for key, value in search.best_params_.items():
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    best_params[key] = value

meta["denial_reason"] = {
    "algorithm": "XGBoost (multi:softprob)",
    "target": "denial_category",
    "best_params": best_params,
    "cv_best_macro_f1": round(float(search.best_score_), 4),
    "metrics": metrics,
    "classes": classes,
    "features_used": FEATURES,
    "top_features": imp_df.head(15)["feature"].tolist(),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
}

meta_path.write_text(json.dumps(meta, indent=2))
print("Saved", MODELS / "denial_reason_model.pkl")
print("=== MODEL 3 COMPLETE ===")


Saved d:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor\models\denial_reason_model.pkl
=== MODEL 3 COMPLETE ===
